## ResNet18 CNN 학습
### 사용데이터
- K001~K006 정상데이터
- KA01 KA03~KA07 이상데이터
- 이상 데이터 셋은 정상데이터와 이상데이터를 섞어서 사용

### 사용 모듈

- `numpy`: STFT `.npy` 배열 처리
- `pandas`: 6축 데이터를 표 형태로 정리
- `pathlib.Path`: 파일 및 디렉터리 경로 처리
- `re`: 파일명 패턴 분석
- `collections.defaultdict`: 항목별 데이터 자동 그룹화
- `sklearn.model_selection.train_test_split`: 데이터를 학습·검증·테스트용으로 계층화 분할
- `torch`: NumPy 배열을 PyTorch Tensor로 변환하고 모델 학습에 사용
- `torch.utils.data.Dataset`: `.npz` 데이터 로딩 방식 정의
- `torch.utils.data.DataLoader`: 데이터를 미니배치 단위로 제공
- `torch.nn`: 신경망 계층과 모델 구조 정의
- `torchvision.models.resnet18`: 축별 특징을 추출하는 ResNet18 모델 생성

In [35]:
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18

### 6축 STFT 데이터 결합

`normal_image`와 `abnormal_image` 폴더에서 STFT `.npy` 파일을 불러오고, 파일명에서 축 이름을 제거한 값을 공통 sample key로 사용합니다. 정상 샘플은 `K001`~`K006`의 6개 정상축으로 구성하고 라벨을 모두 0으로 지정합니다. 이상 샘플은 정상축 5개와 대응하는 이상축 1개를 혼합하며, 이상 위치만 1인 6축 라벨을 생성합니다. 각 배열은 `(F, T, 3)`이면 `(3, F, T)`로 변환하고, 최종 데이터와 라벨을 `dataset/normal`, `dataset/abnormal`에 `.npz`로 저장합니다.

In [ ]:
# 정상축과 대응하는 이상축 설정
normal_axes = ["K001", "K002", "K003", "K004", "K005", "K006"]
abnormal_axes = ["KA01", "KA03", "KA04", "KA05", "KA06", "KA07"]

# 폴더별 STFT 데이터를 sample key로 묶기
def load_axis_data(folder, axis_names):
    grouped = defaultdict(dict)

    # 폴더 안의 모든 .npy 파일 확인
    for file_path in sorted(Path(folder).glob("*.npy")):
        # 파일명에서 축 이름 추출
        match = re.search(r"_(K[A-Z0-9]+)_", file_path.name)
        if match is None:
            continue

        axis_name = match.group(1)
        if axis_name not in axis_names:
            continue

        # 축 이름을 제외한 공통 sample key 생성
        sample_key = file_path.stem.replace(f"_{axis_name}_", "_")

        # STFT 배열 불러오기
        data = np.load(file_path)

        # (F, T, 3)을 (3, F, T)으로 변환
        if data.ndim == 3 and data.shape[-1] == 3:
            data = data.transpose(2, 0, 1)

        # 같은 sample key의 데이터를 축 이름별로 저장
        grouped[sample_key][axis_name] = data.astype(np.float32)

    return grouped

# 필요한 축이 모두 있는지 확인
def has_all_axes(group, axis_names):
    return all(axis_name in group for axis_name in axis_names)

normal_grouped = load_axis_data("normal_image", normal_axes)
abnormal_grouped = load_axis_data("abnormal_image", abnormal_axes)

normal_samples = []
abnormal_samples = []

# 정상 6축 샘플 생성
for sample_key in sorted(normal_grouped):
    normal_data = normal_grouped[sample_key]
    if not has_all_axes(normal_data, normal_axes):
        continue

    normal_arrays = [normal_data[axis_name] for axis_name in normal_axes]
    normal_stack = np.stack(normal_arrays)

    normal_samples.append({
        "sample_key": sample_key,
        "data": normal_stack,
        "label": np.zeros(6, dtype=np.int64),
    })

    # 같은 sample key의 이상축 데이터 확인
    if sample_key not in abnormal_grouped:
        continue

    abnormal_data = abnormal_grouped[sample_key]
    if not has_all_axes(abnormal_data, abnormal_axes):
        continue

    # sample key마다 하나의 이상축을 순환 선택하여 혼합
    axis_index = len(abnormal_samples) % 6
    mixed_stack = normal_stack.copy()
    mixed_stack[axis_index] = abnormal_data[abnormal_axes[axis_index]]

    label = np.zeros(6, dtype=np.int64)
    label[axis_index] = 1

    abnormal_samples.append({
        "sample_key": sample_key,
        "data": mixed_stack,
        "label": label,
    })

# 기존 데이터 확인 코드와 호환되는 배열 목록 생성
normal_stacked_data = [sample["data"] for sample in normal_samples]
abnormal_stacked_data = [sample["data"] for sample in abnormal_samples]

# 생성한 샘플을 .npz 파일로 저장
normal_save_dir = Path("ResNet18\normal_data")
abnormal_save_dir = Path("ResNet18\abnormal_data")
normal_save_dir.mkdir(parents=True, exist_ok=True)
abnormal_save_dir.mkdir(parents=True, exist_ok=True)

for sample in normal_samples:
    save_path = normal_save_dir / f"{sample['sample_key']}.npz"
    np.savez_compressed(save_path, data=sample["data"], label=sample["label"])

for sample in abnormal_samples:
    abnormal_axis = int(np.argmax(sample["label"])) + 1
    save_path = abnormal_save_dir / f"{sample['sample_key']}_axis{abnormal_axis}.npz"
    np.savez_compressed(save_path, data=sample["data"], label=sample["label"])

# 처리 결과 출력
print("Normal samples:", len(normal_stacked_data))
print("Normal first shape:", normal_stacked_data[0].shape if normal_stacked_data else None)
print("Normal first label:", normal_samples[0]["label"] if normal_samples else None)
print("Abnormal samples:", len(abnormal_stacked_data))
print("Abnormal first shape:", abnormal_stacked_data[0].shape if abnormal_stacked_data else None)
print("Abnormal first label:", abnormal_samples[0]["label"] if abnormal_samples else None)

Normal samples: 318
Normal first shape: (6, 3, 1025, 64)
Normal first label: [0 0 0 0 0 0]
Abnormal samples: 308
Abnormal first shape: (6, 3, 1025, 64)
Abnormal first label: [1 0 0 0 0 0]


### 데이터 형태 확인

In [37]:
print("Normal dtype:", normal_stacked_data[0].dtype)
print("Normal finite:", np.isfinite(normal_stacked_data[0]).all())

print("Abnormal dtype:", abnormal_stacked_data[0].dtype)
print("Abnormal finite:", np.isfinite(abnormal_stacked_data[0]).all())

Normal dtype: float32
Normal finite: True
Abnormal dtype: float32
Abnormal finite: True


### 학습·검증·테스트 데이터 분할

`dataset/normal`과 `dataset/abnormal`의 `.npz` 파일명에서 chunk 번호와 이상축 표시를 제거하여 실험 ID를 추출합니다. 먼저 고유한 실험 ID를 학습용 70%, 검증용 15%, 테스트용 15%로 무작위 분할한 뒤, 각 실험에 속한 정상·이상 파일과 모든 chunk를 같은 분할에 배정합니다. 따라서 동일 실험의 chunk가 학습과 검증 또는 테스트에 동시에 포함되는 데이터 누수를 방지합니다. random seed를 고정하여 항상 같은 결과를 만들고, 분할된 파일 경로와 라벨은 이후 데이터 로딩에 사용할 변수로 저장합니다.

In [ ]:
# 정상 및 이상 .npz 파일 경로 불러오기
normal_file_paths = sorted(Path("ResNet18\normal_data").glob("*.npz"))
abnormal_file_paths = sorted(Path("ResNet18\abnormal_data").glob("*.npz"))

if not normal_file_paths or not abnormal_file_paths:
    raise FileNotFoundError("먼저 6축 데이터 생성 및 저장 셀을 실행하세요.")

# 정상은 0, 이상은 1로 분할 확인용 라벨 생성
file_paths = normal_file_paths + abnormal_file_paths
class_labels = np.array(
    [0] * len(normal_file_paths) + [1] * len(abnormal_file_paths),
    dtype=np.int64,
)

# 파일명에서 chunk와 이상축 표시를 제거하여 실험 ID 추출
def get_experiment_id(file_path):
    name = re.sub(r"_axis[1-6]$", "", file_path.stem)
    return re.sub(r"_chunk\d+$", "", name)

experiment_ids = np.array([get_experiment_id(path) for path in file_paths])
unique_experiment_ids = np.array(sorted(set(experiment_ids)))

# 고유 실험 ID를 학습 70%와 임시 30%로 분할
random_seed = 42
train_experiment_ids, temp_experiment_ids = train_test_split(
    unique_experiment_ids,
    test_size=0.30,
    random_state=random_seed,
)

# 임시 실험 ID를 검증 15%와 테스트 15%로 분할
validation_experiment_ids, test_experiment_ids = train_test_split(
    temp_experiment_ids,
    test_size=0.50,
    random_state=random_seed,
)

# 선택된 실험 ID에 속한 파일 경로와 라벨 배정
def collect_split(selected_experiment_ids):
    selected_experiment_ids = set(selected_experiment_ids)
    selected_paths = []
    selected_labels = []

    for path, label, experiment_id in zip(file_paths, class_labels, experiment_ids):
        if experiment_id in selected_experiment_ids:
            selected_paths.append(path)
            selected_labels.append(label)

    return selected_paths, np.array(selected_labels, dtype=np.int64)

train_file_paths, train_labels = collect_split(train_experiment_ids)
validation_file_paths, validation_labels = collect_split(validation_experiment_ids)
test_file_paths, test_labels = collect_split(test_experiment_ids)

# 분할 사이에 동일한 실험 ID가 없는지 확인
train_id_set = set(train_experiment_ids)
validation_id_set = set(validation_experiment_ids)
test_id_set = set(test_experiment_ids)

assert train_id_set.isdisjoint(validation_id_set)
assert train_id_set.isdisjoint(test_id_set)
assert validation_id_set.isdisjoint(test_id_set)

# 각 분할의 실험 수와 정상·이상 샘플 수 출력
def print_split_count(name, experiment_ids, labels):
    normal_count = np.sum(labels == 0)
    abnormal_count = np.sum(labels == 1)
    print(
        f"{name} - Experiments: {len(experiment_ids)}, "
        f"Normal: {normal_count}, Abnormal: {abnormal_count}"
    )

print_split_count("Train", train_experiment_ids, train_labels)
print_split_count("Validation", validation_experiment_ids, validation_labels)
print_split_count("Test", test_experiment_ids, test_labels)

Train - Experiments: 56, Normal: 223, Abnormal: 218
Validation - Experiments: 12, Normal: 47, Abnormal: 45
Test - Experiments: 12, Normal: 48, Abnormal: 45


### 분할 결과 누락 확인

In [39]:
# 파일 경로 중 이상 라벨인 경로만 선택
def select_abnormal_paths(paths, labels):
    return [path for path, label in zip(paths, labels) if label == 1]

train_abnormal_file_paths = select_abnormal_paths(train_file_paths, train_labels)
validation_abnormal_file_paths = select_abnormal_paths(
    validation_file_paths, validation_labels
)
test_abnormal_file_paths = select_abnormal_paths(test_file_paths, test_labels)

# 원본과 분할된 이상 데이터 수 비교
print("Original abnormal:", len(abnormal_file_paths))
all_split_abnormal_file_paths = (
    train_abnormal_file_paths
    + validation_abnormal_file_paths
    + test_abnormal_file_paths
)
print("Split abnormal:", len(all_split_abnormal_file_paths))

missing_abnormal_file_paths = set(abnormal_file_paths) - set(all_split_abnormal_file_paths)

print("Missing abnormal:", len(missing_abnormal_file_paths))
print(*sorted(missing_abnormal_file_paths), sep="\n")

Original abnormal: 308
Split abnormal: 308
Missing abnormal: 0



### PyTorch Dataset과 DataLoader 구성

학습용, 검증용, 테스트용으로 분할된 `.npz` 파일 경로를 PyTorch `Dataset`으로 구성합니다. 먼저 학습 데이터만 사용하여 6개 축과 3개 채널 각각의 평균과 표준편차를 계산합니다. 모든 샘플에 동일한 학습 통계를 적용하므로 데이터 누수를 방지하면서 샘플 간 상대적인 크기 차이를 유지할 수 있습니다. 각 파일은 마지막 주파수 bin을 제거해 `(6, 3, 1024, 64)`로 맞춘 뒤 `(data - mean) / std`로 표준화하고 `float32` Tensor로 변환합니다. 학습 DataLoader에만 `shuffle=True`를 적용합니다.

In [40]:
# 학습 데이터의 축·채널별 평균과 표준편차 계산
def calculate_normalization_stats(file_paths):
    value_sum = np.zeros((6, 3), dtype=np.float64)
    square_sum = np.zeros((6, 3), dtype=np.float64)
    value_count = 0

    for file_path in file_paths:
        with np.load(file_path) as sample:
            data = sample["data"][:, :, :-1, :]

        value_sum += data.sum(axis=(2, 3), dtype=np.float64)
        square_sum += (data * data).sum(axis=(2, 3), dtype=np.float64)
        value_count += data.shape[2] * data.shape[3]

    mean = value_sum / value_count
    variance = square_sum / value_count - mean ** 2
    std = np.sqrt(np.maximum(variance, 1e-12))

    mean = mean.astype(np.float32)[:, :, None, None]
    std = std.astype(np.float32)[:, :, None, None]
    return mean, std

normalization_mean, normalization_std = calculate_normalization_stats(train_file_paths)

# .npz 파일에서 데이터와 라벨을 불러오는 Dataset
class STFTDataset(Dataset):
    def __init__(self, file_paths, mean, std):
        self.file_paths = list(file_paths)
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, index):
        file_path = self.file_paths[index]

        with np.load(file_path) as sample:
            data = sample["data"]
            label = sample["label"]

        if data.shape != (6, 3, 1025, 64):
            raise ValueError(f"Unexpected data shape {data.shape}: {file_path}")

        # 마지막 주파수 bin을 제거하여 (6, 3, 1024, 64)로 변환
        data = data[:, :, :-1, :]

        # 학습 데이터 통계로 축·채널별 표준화
        data = (data - self.mean) / self.std

        data_tensor = torch.from_numpy(data.astype(np.float32, copy=False))
        label_tensor = torch.from_numpy(label.astype(np.float32, copy=False))

        return data_tensor, label_tensor

# 분할별 Dataset 생성
train_dataset = STFTDataset(train_file_paths, normalization_mean, normalization_std)
validation_dataset = STFTDataset(
    validation_file_paths, normalization_mean, normalization_std
)
test_dataset = STFTDataset(test_file_paths, normalization_mean, normalization_std)

# 분할별 DataLoader 생성
batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 첫 번째 학습 배치의 shape 확인
first_batch_data, first_batch_labels = next(iter(train_loader))
print("Batch data shape:", first_batch_data.shape)
print("Batch label shape:", first_batch_labels.shape)
print("Batch mean:", first_batch_data.mean().item())
print("Batch std:", first_batch_data.std().item())

Batch data shape: torch.Size([8, 6, 3, 1024, 64])
Batch label shape: torch.Size([8, 6])
Batch mean: 0.014722632244229317
Batch std: 1.0182442665100098


### 축 간 관계를 학습하는 공유 ResNet18 모델

하나의 ResNet18을 6개 축에 공유하여 각 축에서 512차원 특징을 추출합니다. 작은 배치와 축별 순차 처리에서도 정규화 통계가 안정적으로 유지되도록 ResNet18의 BatchNorm 대신 GroupNorm을 사용합니다. 추출된 특징은 `(B, 6, 512)`로 쌓은 뒤 128차원으로 축소하고, 학습 가능한 축 위치 정보와 함께 self-attention에 전달합니다. self-attention은 한 축을 판단할 때 다른 5개 축의 특징을 함께 참고하여 축 간 상대적인 차이와 공통 패턴을 학습합니다. 관계가 반영된 각 축 특징을 공유 분류층에 전달하여 최종 `(B, 6)` logits를 출력하며, sigmoid를 적용하면 축별 이상 확률을 얻을 수 있습니다.

In [41]:
# 축 간 관계를 함께 학습하는 공유 ResNet18 모델
class SixAxisResNet18(nn.Module):
    def __init__(self):
        super().__init__()

        # 6개 축이 공유하는 ResNet18 특징 추출기 생성
        self.backbone = resnet18(
            weights=None,
            norm_layer=lambda channels: nn.GroupNorm(32, channels),
        )

        # 마지막 분류층을 제거하여 512차원 특징 출력
        self.backbone.fc = nn.Identity()

        # 축별 특징을 관계 학습에 사용할 128차원으로 축소
        self.feature_projection = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.LayerNorm(128),
        )

        # 모델이 6개 축의 위치를 구분하도록 축 위치 정보 생성
        self.axis_embedding = nn.Parameter(torch.zeros(1, 6, 128))
        nn.init.normal_(self.axis_embedding, std=0.02)

        # Self-Attention으로 6개 축 사이의 관계 학습
        relation_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=4,
            dim_feedforward=256,
            dropout=0.2,
            batch_first=True,
            norm_first=True,
        )
        self.relation_encoder = nn.TransformerEncoder(relation_layer, num_layers=1)

        # 관계가 반영된 각 축 특징을 하나의 logit으로 변환
        self.classifier = nn.Linear(128, 1)

    def forward(self, x):
        axis_features = []

        # 각 축을 동일한 ResNet18에 입력하여 특징 추출
        for axis_index in range(6):
            features = self.backbone(x[:, axis_index])
            axis_features.append(features)

        # 축별 특징을 (B, 6, 512) 형태로 결합
        axis_features = torch.stack(axis_features, dim=1)

        # 특징 차원 축소 후 축 위치 정보 추가
        relation_features = self.feature_projection(axis_features)
        relation_features = relation_features + self.axis_embedding

        # 다른 축의 특징을 참고하여 축별 관계 특징 생성
        relation_features = self.relation_encoder(relation_features)

        # 각 축의 이상 판단을 위한 (B, 6) logits 출력
        logits = self.classifier(relation_features).squeeze(-1)

        return logits

# GPU 사용 가능 여부에 따라 device 설정

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SixAxisResNet18().to(device)

# 첫 번째 학습 배치를 모델에 입력하여 shape 확인
batch_data, batch_labels = next(iter(train_loader))
model.eval()

with torch.no_grad():
    batch_logits = model(batch_data.to(device))

print("Device:", device)
print("Input shape:", batch_data.shape)
print("Output shape:", batch_logits.shape)

/tmp/ipykernel_268916/3642029259.py:35: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.relation_encoder = nn.TransformerEncoder(relation_layer, num_layers=1)


Device: cuda
Input shape: torch.Size([8, 6, 3, 1024, 64])
Output shape: torch.Size([8, 6])


### 6축 다중 라벨 모델 학습과 검증

각 샘플의 6개 축은 서로 독립적인 정상·이상 라벨을 가지므로 `BCEWithLogitsLoss`를 사용합니다. 축 단위 이상 라벨이 적은 불균형을 보정하기 위해 학습 분할에서 축별 정상·이상 개수를 계산하고, `정상 개수 / 이상 개수`를 각 축의 `pos_weight`로 적용합니다. 따라서 이상축을 놓쳤을 때의 손실을 더 크게 반영하며 validation/test 라벨은 가중치 계산에 사용하지 않습니다. Adam 옵티마이저로 여러 epoch 동안 학습하고, 검증에서는 gradient를 계산하지 않습니다. epoch마다 평균 손실을 출력하고 검증 손실이 가장 낮은 모델 가중치를 저장합니다.

In [42]:
# 학습 파일에서 6축 라벨만 불러오기
train_axis_labels = []
for file_path in train_file_paths:
    with np.load(file_path) as sample:
        train_axis_labels.append(sample["label"])

train_axis_labels = np.stack(train_axis_labels)

# 축별 정상·이상 개수로 positive class 가중치 계산
positive_counts = train_axis_labels.sum(axis=0)
negative_counts = len(train_axis_labels) - positive_counts

if np.any(positive_counts == 0):
    raise ValueError("학습 데이터에 이상 샘플이 없는 축이 있습니다.")

pos_weight = torch.tensor(
    negative_counts / positive_counts,
    dtype=torch.float32,
    device=device,
)

print("Positive counts:", positive_counts.astype(int))
print("Negative counts:", negative_counts.astype(int))
print("Positive weights:", pos_weight.cpu().numpy())

# 가중 손실 함수와 옵티마이저 설정
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 10
best_validation_loss = float("inf")
best_model_path = Path("best_six_axis_resnet18.pth")

# 여러 epoch 동안 학습과 검증 수행
for epoch in range(num_epochs):
    model.train()
    train_loss_sum = 0.0

    # 학습 단계
    for batch_data, batch_labels in train_loader:
        batch_data = batch_data.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()
        logits = model(batch_data)
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * batch_data.size(0)

    train_loss = train_loss_sum / len(train_dataset)

    model.eval()
    validation_loss_sum = 0.0

    # 검증 단계
    with torch.no_grad():
        for batch_data, batch_labels in validation_loader:
            batch_data = batch_data.to(device)
            batch_labels = batch_labels.to(device)

            logits = model(batch_data)
            loss = criterion(logits, batch_labels)
            validation_loss_sum += loss.item() * batch_data.size(0)

    validation_loss = validation_loss_sum / len(validation_dataset)

    print(
        f"Epoch {epoch + 1}/{num_epochs} - "
        f"Train Loss: {train_loss:.4f}, "
        f"Validation Loss: {validation_loss:.4f}"
    )

    # 가장 낮은 validation loss의 모델 가중치 저장
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"Best model saved: {best_model_path}")

Positive counts: [38 37 37 34 35 37]
Negative counts: [403 404 404 407 406 404]
Positive weights: [10.605263 10.918919 10.918919 11.970589 11.6      10.918919]
Epoch 1/10 - Train Loss: 1.3170, Validation Loss: 1.2688
Best model saved: best_six_axis_resnet18.pth
Epoch 2/10 - Train Loss: 1.2894, Validation Loss: 1.2645
Best model saved: best_six_axis_resnet18.pth
Epoch 3/10 - Train Loss: 1.2770, Validation Loss: 1.1729
Best model saved: best_six_axis_resnet18.pth
Epoch 4/10 - Train Loss: 0.9958, Validation Loss: 0.9103
Best model saved: best_six_axis_resnet18.pth
Epoch 5/10 - Train Loss: 0.7331, Validation Loss: 0.4320
Best model saved: best_six_axis_resnet18.pth
Epoch 6/10 - Train Loss: 0.4686, Validation Loss: 0.9756
Epoch 7/10 - Train Loss: 0.4361, Validation Loss: 0.3032
Best model saved: best_six_axis_resnet18.pth
Epoch 8/10 - Train Loss: 0.3268, Validation Loss: 0.2848
Best model saved: best_six_axis_resnet18.pth
Epoch 9/10 - Train Loss: 0.2584, Validation Loss: 0.4214
Epoch 10/10 